In [11]:
import torch
from torchvision.io import read_video
from torchvision.models.video import r3d_18, R3D_18_Weights
import glob
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
test_list_pass = glob.glob('./data/test/no_shot/*.mp4')
labels_test_pass = [0 for i in range(len(test_list_pass))]

test_list_shot = glob.glob('./data/test/shot/*.mp4')
labels_test_shot = [1 for i in range(len(test_list_shot))]

videos_test = test_list_pass + test_list_shot
labels_test = labels_test_pass + labels_test_shot

print(f'Test videos {len(videos_test)}')

Test videos 68


# r3d_18

In [4]:
print("Loading pre-trained ResNet3D-18 model")
weights = R3D_18_Weights.DEFAULT
model = r3d_18(weights=weights)
model = model.to(DEVICE)
model.eval()
kinetics_labels = weights.meta["categories"]
preprocess = weights.transforms()

Loading pre-trained ResNet3D-18 model


In [5]:
predicted_labels_1 = []
predicted_labels_5 = []
video_number = 1
total_videos = len(videos_test)
for (VIDEO_PATH, label) in zip(videos_test, labels_test):
    print(f"\nLoading video from: {VIDEO_PATH}")
    print(f'Video {video_number} out of {total_videos}')
    video_number += 1
    try:
        video_frames, audio, info = read_video(VIDEO_PATH, output_format="TCHW", pts_unit="sec")
        print(f"Original Video shape: {video_frames.shape}")
                
        num_frames = video_frames.shape[0]  
        desired_frames = 16
        
        if num_frames >= desired_frames:
            indices = torch.linspace(0, num_frames - 1, steps=desired_frames).long()
            video_sampled = video_frames[indices]
        else:
            padding_size = desired_frames - num_frames
            last_frame = video_frames[-1:]
            padding = last_frame.repeat(padding_size, 1, 1, 1)
            video_sampled = torch.cat([video_frames, padding], dim=0)

        print(f"Sampled Video shape (T, C, H, W): {video_sampled.shape}")

        video_preprocessed = preprocess(video_sampled)
        print(f"Preprocessed video shape: {video_preprocessed.shape}")
        
        if video_preprocessed.dim() == 4:
            video_preprocessed = video_preprocessed.unsqueeze(0)

        with torch.no_grad():
            outputs = model(video_preprocessed.to(DEVICE))
        
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        top5_probs, top5_indices = torch.topk(probabilities, 5, dim=1)
        top5_labels = top5_indices.flatten().cpu().tolist()
        if top5_labels[0] == 296:
            predicted_labels_1.append(1)
        else:
            predicted_labels_1.append(0)
        if 296 in top5_labels:
            predicted_labels_5.append(1)
        else:
            predicted_labels_5.append(0)
        #print(f'Class "{kinetics_labels[top1_indices.item()]}" with probability {top1_probs.item() * 100:.2f}%')

    except FileNotFoundError:
        print(f"Error: Video file '{VIDEO_PATH}' not found!")
    except Exception as e:
        print(f"Error during processing: {type(e).__name__}: {e}")


Loading video from: ./data/test/no_shot\107.mp4
Video 1 out of 68
Original Video shape: torch.Size([417, 3, 720, 1280])
Sampled Video shape (T, C, H, W): torch.Size([16, 3, 720, 1280])
Preprocessed video shape: torch.Size([3, 16, 112, 112])

Loading video from: ./data/test/no_shot\116.mp4
Video 2 out of 68
Original Video shape: torch.Size([471, 3, 720, 1280])
Sampled Video shape (T, C, H, W): torch.Size([16, 3, 720, 1280])
Preprocessed video shape: torch.Size([3, 16, 112, 112])

Loading video from: ./data/test/no_shot\121.mp4
Video 3 out of 68
Original Video shape: torch.Size([689, 3, 720, 1280])
Sampled Video shape (T, C, H, W): torch.Size([16, 3, 720, 1280])
Preprocessed video shape: torch.Size([3, 16, 112, 112])

Loading video from: ./data/test/no_shot\133.mp4
Video 4 out of 68
Original Video shape: torch.Size([513, 3, 720, 1280])
Sampled Video shape (T, C, H, W): torch.Size([16, 3, 720, 1280])
Preprocessed video shape: torch.Size([3, 16, 112, 112])

Loading video from: ./data/test

In [14]:
print(confusion_matrix(labels_test, predicted_labels_1))
print(confusion_matrix(labels_test, predicted_labels_5))

[[34  0]
 [34  0]]
[[ 9 25]
 [ 7 27]]


# SlowFast

In [ ]:
import torch
from torchvision.io import read_video
from torchvision.transforms import Compose, Lambda
from torchvision.transforms._transforms_video import CenterCropVideo, NormalizeVideo
import glob 
import sys
from torchvision.transforms import functional as F
sys.modules["torchvision.transforms.functional_tensor"] = F
from pytorchvideo.transforms import ShortSideScale, UniformTemporalSubsample
print(f'Test videos {len(videos_test)}')


print("Loading pre-trained SlowFast_R50 model from PyTorch Hub")
model = torch.hub.load("facebookresearch/pytorchvideo", model="slowfast_r50", pretrained=True)
model = model.to(DEVICE)
model.eval()

# Kinetics-400 configuration defaults for SlowFast
side_size = 256
crop_size = 256
mean = [0.45, 0.45, 0.45]
std = [0.225, 0.225, 0.225]
num_frames = 32  # Fast pathway samples 32 frames
alpha = 4        # Slow pathway samples 1/4 of the frames (8 frames)

class PackPathway(torch.nn.Module):
    """Splits a single video tensor into [Slow_Pathway, Fast_Pathway]"""
    def __init__(self):
        super().__init__()
    def forward(self, frames: torch.Tensor):
        fast_pathway = frames
        # Temporal sampling for the slow pathway
        slow_pathway = torch.index_select(
            frames,
            1, # dimension 1 is temporal (T) after permute
            torch.linspace(0, frames.shape[1] - 1, frames.shape[1] // alpha).long(),
        )
        return [slow_pathway, fast_pathway]

# Build custom transform pipeline mirroring the native weights requirements
preprocess = Compose([
    UniformTemporalSubsample(num_frames),
    Lambda(lambda x: x / 255.0), # Normalize pixel values to [0, 1]
    NormalizeVideo(mean, std),
    ShortSideScale(size=side_size),
    CenterCropVideo(crop_size),
    PackPathway()
])

predicted_labels_1 = []
predicted_labels_5 = []
video_number = 1
total_videos = len(videos_test)

for (VIDEO_PATH, label) in zip(videos_test, labels_test):
    print(f"\nLoading video from: {VIDEO_PATH}")
    print(f'Video {video_number} out of {total_videos}')
    video_number += 1
    try:
        # read_video outputs (T, H, W, C) by default. 
        # PyTorchVideo transforms expect (C, T, H, W)
        video_frames, audio, info = read_video(VIDEO_PATH, pts_unit="sec")
        video_frames = video_frames.permute(3, 0, 1, 2) # Shift to (C, T, H, W)
        print(f"Original Video shape (C, T, H, W): {video_frames.shape}")
                
        num_frames_raw = video_frames.shape[1]  
        
        # Ensure we have at least 32 frames to cleanly sub-sample
        if num_frames_raw < num_frames:
            padding_size = num_frames - num_frames_raw
            last_frame = video_frames[:, -1:]
            padding = last_frame.repeat(1, padding_size, 1, 1)
            video_frames = torch.cat([video_frames, padding], dim=1)

        # 3. Apply custom preprocessing split
        video_preprocessed = preprocess(video_frames)
        
        # 4. Add batch dimension and send to device for both tensors
        video_inputs = [tensor.unsqueeze(0).to(DEVICE) for tensor in video_preprocessed]
        
        print(f"Slow pathway shape: {video_inputs[0].shape}") # Expects: (1, 3, 8, 256, 256)
        print(f"Fast pathway shape: {video_inputs[1].shape}") # Expects: (1, 3, 32, 256, 256)

        with torch.no_grad():
            outputs = model(video_inputs)
        
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        top5_probs, top5_indices = torch.topk(probabilities, 5, dim=1)
        top5_labels = top5_indices.flatten().cpu().tolist()
        
        if top5_labels[0] == 296:
            predicted_labels_1.append(1)
        else:
            predicted_labels_1.append(0)
            
        if 296 in top5_labels:
            predicted_labels_5.append(1)
        else:
            predicted_labels_5.append(0)

    except FileNotFoundError:
        print(f"Error: Video file '{VIDEO_PATH}' not found!")
    except Exception as e:
        print(f"Error during processing: {type(e).__name__}: {e}")

Test videos 68
Loading pre-trained SlowFast_R50 model from PyTorch Hub


Using cache found in C:\Users\hecto/.cache\torch\hub\facebookresearch_pytorchvideo_main



Loading video from: ./data/test/no_shot\107.mp4
Video 1 out of 68
Original Video shape (C, T, H, W): torch.Size([3, 417, 720, 1280])
Slow pathway shape: torch.Size([1, 3, 8, 256, 256])
Fast pathway shape: torch.Size([1, 3, 32, 256, 256])

Loading video from: ./data/test/no_shot\116.mp4
Video 2 out of 68
Original Video shape (C, T, H, W): torch.Size([3, 471, 720, 1280])
Slow pathway shape: torch.Size([1, 3, 8, 256, 256])
Fast pathway shape: torch.Size([1, 3, 32, 256, 256])

Loading video from: ./data/test/no_shot\121.mp4
Video 3 out of 68
Original Video shape (C, T, H, W): torch.Size([3, 689, 720, 1280])
Slow pathway shape: torch.Size([1, 3, 8, 256, 256])
Fast pathway shape: torch.Size([1, 3, 32, 256, 256])

Loading video from: ./data/test/no_shot\133.mp4
Video 4 out of 68
Original Video shape (C, T, H, W): torch.Size([3, 513, 720, 1280])
Slow pathway shape: torch.Size([1, 3, 8, 256, 256])
Fast pathway shape: torch.Size([1, 3, 32, 256, 256])

Loading video from: ./data/test/no_shot\149

In [19]:
print(confusion_matrix(labels_test, predicted_labels_1))
print(confusion_matrix(labels_test, predicted_labels_5))

[[34  0]
 [33  1]]
[[ 0 34]
 [ 0 34]]


# VideoMAE